# Feature Engineering

Load the processed base dataset (fresh YFinance prices, target defined), re-derive all WTI/Brent features, and build three model-ready datasets: full, core, and PCA.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from pathlib import Path

PROCESSED_DIR = Path("../..") / "data" / "processed"

df = pd.read_parquet(PROCESSED_DIR / "processed_base.parquet")
df["Date"] = pd.to_datetime(df["Date"])
print(f"Loaded processed_base: {df.shape}")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")

## Derive WTI & Brent features from fresh prices

In [ ]:
wti_price = df["WTI_Close"]
brent_price = df["Brent_Close"]

derived = pd.DataFrame(index=df.index)

# Log returns
derived["WTI_Log_Returns"] = np.log(wti_price / wti_price.shift(1))
derived["Brent_Log_Return"] = np.log(brent_price / brent_price.shift(1))

# Lagged WTI returns (1-10)
for lag in range(1, 11):
    derived[f"WTI_Ret_Lag{lag}"] = derived["WTI_Log_Returns"].shift(lag)

# Rolling volatility
for w in [5, 10, 20, 60]:
    derived[f"WTI_Ret_Vol_{w}D"] = derived["WTI_Log_Returns"].rolling(w).std()

# Spot momentum
for w in [5, 20]:
    derived[f"WTI_Spot_Momentum_{w}D"] = wti_price.pct_change(w) * 100

# Brent-WTI spread
derived["Brent_WTI_Spread"] = brent_price - wti_price
derived["Brent_WTI_Spread_Chg"] = derived["Brent_WTI_Spread"].diff()

# Insert after Date, Target, WTI_Close, Brent_Close (columns 0-3)
other_cols = [c for c in df.columns if c not in ["Date", "Target", "WTI_Close", "Brent_Close"]]
df = pd.concat([df[["Date", "Target", "WTI_Close", "Brent_Close"]], derived, df[other_cols]], axis=1)

# Drop rows with NaN
rows_before = len(df)
df = df.dropna().reset_index(drop=True)
print(f"Dropped {rows_before - len(df)} NaN rows")
print(f"Full dataset: {df.shape}")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"\nDerived features ({len(derived.columns)}):")
for c in derived.columns:
    print(f"  {c}")

## Build Core dataset

WTI and Brent returns with lags 1-5, plus WTI volatility.

In [ ]:
CORE_FEATURES = [
    "WTI_Log_Returns",
    "Brent_Log_Return",
    "SP500_Log_Return",
    "VIX_Log_Return",
    "DXY_Log_Return",
    "WTI_Ret_Lag1", "WTI_Ret_Lag2", "WTI_Ret_Lag3", "WTI_Ret_Lag4", "WTI_Ret_Lag5",
    "WTI_Ret_Vol_5D", "WTI_Ret_Vol_10D", "WTI_Ret_Vol_20D",
]

keep_core = ["Date", "Target", "WTI_Close", "Brent_Close"] + CORE_FEATURES
df_core = df[keep_core].copy()

print(f"Core dataset: {df_core.shape}")
print(f"  Features: {len(CORE_FEATURES)}")
print(f"  Columns: {list(df_core.columns)}")

## Build PCA dataset

Same as full but the 28 Fed text columns are compressed into PCA components (80% variance).

In [ ]:
fed_cols = [c for c in df.columns if c.startswith("Fed")]
non_fed_cols = [c for c in df.columns if not c.startswith("Fed")]

print(f"Fed columns to compress: {len(fed_cols)}")

# Fit PCA on Fed columns
fed_matrix = df[fed_cols].ffill().bfill()
scaler = StandardScaler()
fed_std = scaler.fit_transform(fed_matrix)

pca = PCA(random_state=42)
pca.fit(fed_std)
cumvar = np.cumsum(pca.explained_variance_ratio_)
n_components = int(np.searchsorted(cumvar, 0.80) + 1)

pca_final = PCA(n_components=n_components, random_state=42)
pc_scores = pca_final.fit_transform(fed_std)

pc_cols = [f"Fed_PC{i+1}" for i in range(n_components)]
pc_df = pd.DataFrame(pc_scores, columns=pc_cols, index=df.index)

df_pca = pd.concat([df[non_fed_cols], pc_df], axis=1)

print(f"PCA dataset: {df_pca.shape}")
print(f"  Fed PCA components: {n_components} ({cumvar[n_components-1]:.1%} variance explained)")
print(f"  Total features: {df_pca.shape[1] - 4}")

## Save all datasets

In [ ]:
for name, data in [("sanity_data", df), ("sanity_data_core", df_core), ("sanity_data_pca", df_pca)]:
    data.to_parquet(PROCESSED_DIR / f"{name}.parquet", index=False)
    data.to_excel(PROCESSED_DIR / f"{name}.xlsx", index=False, sheet_name=name.split("_")[-1])

print("Saved:")
print(f"  sanity_data.parquet       ({df.shape[0]} x {df.shape[1]})")
print(f"  sanity_data_core.parquet  ({df_core.shape[0]} x {df_core.shape[1]})")
print(f"  sanity_data_pca.parquet   ({df_pca.shape[0]} x {df_pca.shape[1]})")
print(f"\nAll datasets saved as .parquet and .xlsx")